# Hyperparameter Tuning - XGBoost
---

En este cuadernillo se realizará una búsqueda de hiperparámetros utilizando `RandomizedSearchCV` para encontrar la mejor combinación de los hiperparámetros del modelo `XGBClassifier`. Se evaluarán diferentes valores para los hiperparámetros `n_estimators`, `max_depth`, `learning_rate`, `min_child_weight`, `subsample` y `colsample_bytree`.

In [4]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Tras instalar las librerías necesarias, se cargarán los datos de entrenamiento y validación, manteniendo el conjunto de prueba totalmente separado para una evaluación realista del modelo final.

In [5]:
import sys
from pathlib import Path

root_path = Path.cwd().parent
sys.path.append(str(root_path))

from src.data import DataLoader

In [6]:
loader = DataLoader()
X_train, y_train, X_val, y_val, X_test, y_test = loader.load_data()

print(f"   Train: {X_train.shape[0]} samples")
print(f"   Val:   {X_val.shape[0]} samples")
print(f"   Test:  {X_test.shape[0]} samples")
print(f"   Classes: {loader.get_class_names()}")

   Train: 580800 samples
   Val:   11448 samples
   Test:  11472 samples
   Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


Posteriormente, se definirá el modelo base de `XGBClassifier` y se configurará el espacio de búsqueda de hiperparámetros. Se ejecutará `RandomizedSearchCV` para encontrar la mejor combinación de hiperparámetros basada en la puntuación de validación.

In [7]:
# Definir el modelo base
xgboost = XGBClassifier()

# Configurar el espacio de búsqueda de hiperparámetros
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0]
}

# Ejecutar RandomizedSearchCV
grid_search = RandomizedSearchCV(estimator=xgboost, param_distributions=param_grid, 
                                n_iter=10, cv=5, n_jobs=-1, scoring='f1_macro')
grid_search.fit(X_train, y_train)

# Imprimir los mejores hiperparámetros
print("Best Hyperparameters:", grid_search.best_params_)

Best Hyperparameters: {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.9}


In [8]:
# Evaluar el modelo con los mejores hiperparámetros en el conjunto de validación
best_xgboost = grid_search.best_estimator_
y_val_pred = best_xgboost.predict(X_val)
print("Classification Report on Validation Set:")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix on Validation Set:")
print(confusion_matrix(y_val, y_val_pred))

Classification Report on Validation Set:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       477
           1       0.99      1.00      1.00       477
           2       0.99      0.99      0.99       477
           3       0.99      0.98      0.98       477
           4       0.99      0.99      0.99       477
           5       0.99      1.00      0.99       477
           6       0.98      0.99      0.99       477
           7       0.98      0.99      0.98       477
           8       0.99      0.97      0.98       477
           9       0.99      0.99      0.99       477
          10       0.99      0.99      0.99       477
          11       0.92      0.98      0.95       477
          12       0.99      0.94      0.96       477
          13       0.98      0.99      0.98       477
          14       0.99      0.99      0.99       477
          15       0.99      0.98      0.99       477
          16       0.98      0.98      0